In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib.pyplot as plt

#Load dataset
dataset = pd.read_csv("Meta Model Dataset/Training_For_Meta_Model.csv")

In [ ]:
lifestyle_columns_to_drop = ['Cholesterol_Level', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure', 'Glucose_Level', 'id']
health_columns_to_drop = ['Smoking_Status', 'Physical_Activity', 'Alcohol_Intake','id']

X = dataset.drop(['Cardiovascular Disease'], axis=1)
Y = dataset['Cardiovascular Disease']


X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2, 
    random_state=42,
    stratify=Y 
)

X_train = X_train.rename(columns={
    'Cholesterol Level': 'Cholesterol_Level',
    'Diastolic Blood Pressure': 'Diastolic_Blood_Pressure',
    'Glucose Level': 'Glucose_Level',
    'Systolic Blood Pressure': 'Systolic_Blood_Pressure',
    'Alcohol Intake': 'Alcohol_Intake',
    'Physical Activity': 'Physical_Activity',
    'Smoking Status': 'Smoking_Status'
})


health_dataset = X_train.drop(columns=health_columns_to_drop)
lifestyle_dataset = X_train.drop(columns=lifestyle_columns_to_drop)

#print(health_dataset.head())
#print(lifestyle_dataset.head())

In [6]:
import lightgbm as lgb

import os

# Load the trained models
model_B = lgb.Booster(model_file=r"saved_models_tausif\lightgbm_model_B.txt")
model_A = lgb.Booster(model_file=r"saved_models_tausif\lightgbm_model_A.txt")

# Predict using the respective datasets
predictions_A = model_A.predict(lifestyle_dataset)
predictions_B = model_B.predict(health_dataset)


print(predictions_A)
print(predictions_B)


[0.51450838 0.66875897 0.62748518 ... 0.35488267 0.33029411 0.32365621]
[0.20400517 0.92270298 0.807062   ... 0.22936528 0.58392303 0.22233091]


CREATE META MODEL

In [ ]:
meta_features = np.column_stack((predictions_A, predictions_B))
#print(meta_features.shape)

meta_target = Y_train.values  # ensure it's a numpy array


(10921,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

meta_model = Sequential([
    Input(shape=(2,)),   # 2 features from base models
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification output
])

meta_model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC()])
meta_model.fit(meta_features, meta_target, epochs=500, batch_size=16, validation_split=0.2)


